In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, roc_curve

print("All libraries loaded!")

All libraries loaded!


In [2]:
df = pd.read_excel("Credit card default subsample.xls", header=1)
print(df.shape)
df.head()

(7000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,14833,120000,2,2,1,41,2,0,0,0,...,44063,46686,48768,4000,2000,3500,3500,3000,0,1
1,15489,200000,1,1,1,42,1,-1,-1,-1,...,581,581,581,581,581,581,581,581,581,1
2,21234,80000,1,3,2,33,1,-1,-1,-1,...,0,0,0,3495,765,0,0,0,0,0
3,19382,90000,2,2,1,37,1,2,2,2,...,20042,20408,22073,0,3300,0,1000,2000,0,0
4,12417,60000,1,2,2,27,1,2,2,2,...,11480,10851,11365,2000,0,1000,0,1000,0,1


In [3]:
print(df.columns.tolist())


['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


In [4]:
print("Data types:")
print(df.dtypes)
print("---")
print("Missing values per column:")
print(df.isnull().sum())

Data types:
ID                            int64
LIMIT_BAL                     int64
SEX                           int64
EDUCATION                     int64
MARRIAGE                      int64
AGE                           int64
PAY_0                         int64
PAY_2                         int64
PAY_3                         int64
PAY_4                         int64
PAY_5                         int64
PAY_6                         int64
BILL_AMT1                     int64
BILL_AMT2                     int64
BILL_AMT3                     int64
BILL_AMT4                     int64
BILL_AMT5                     int64
BILL_AMT6                     int64
PAY_AMT1                      int64
PAY_AMT2                      int64
PAY_AMT3                      int64
PAY_AMT4                      int64
PAY_AMT5                      int64
PAY_AMT6                      int64
default payment next month    int64
dtype: object
---
Missing values per column:
ID                            0
LIMIT_BAL  

In [5]:
# Cell 5: Check unique values in categorical columns
print("SEX:", df['SEX'].unique())
# SEX only has 1 or 2 - clean, no issues

print("EDUCATION:", df['EDUCATION'].unique())
# EDUCATION only has 1/2/3/4, presence of 0/5/6 needs cleaning

print("MARRIAGE:", df['MARRIAGE'].unique())
# MARRIAGE only has 1/2/3, presence of 0 needs cleaning

print("Default:", df['default payment next month'].unique())
# Default only has 0/1 - clean, no issues

print("PAY_0:", df['PAY_0'].unique())
print("PAY_2:", df['PAY_2'].unique())
print("PAY_3:", df['PAY_3'].unique())
print("PAY_4:", df['PAY_4'].unique())
print("PAY_5:", df['PAY_5'].unique())
print("PAY_6:", df['PAY_6'].unique())
# Should only have numbers from -2 to 9, clean

print("LIMIT_BAL min:", df['LIMIT_BAL'].min())
print("LIMIT_BAL max:", df['LIMIT_BAL'].max())
# Positive numbers only, clean

print("AGE min:", df['AGE'].min())
print("AGE max:", df['AGE'].max())
# Reasonable age range, clean

print("PAY_AMT1 min:", df['PAY_AMT1'].min())
print("PAY_AMT2 min:", df['PAY_AMT2'].min())
print("PAY_AMT3 min:", df['PAY_AMT3'].min())
print("PAY_AMT4 min:", df['PAY_AMT4'].min())
print("PAY_AMT5 min:", df['PAY_AMT5'].min())
print("PAY_AMT6 min:", df['PAY_AMT6'].min())
# Should never be negative, clean



SEX: [2 1]
EDUCATION: [2 1 3 5 6 0 4]
MARRIAGE: [1 2 0 3]
Default: [1 0]
PAY_0: [ 2  1  0 -1 -2  3  4  5  8  7  6]
PAY_2: [ 0 -1  2  3 -2  4  5  1  7  6]
PAY_3: [ 0 -1  2 -2  3  4  5  6  8  7  1]
PAY_4: [ 0 -1  2 -2  3  7  4  5  8  1]
PAY_5: [ 0 -1 -2  2  3  7  5  4  8  6]
PAY_6: [ 2 -1 -2  0  3  7  6  5  4  8]
LIMIT_BAL min: 10000
LIMIT_BAL max: 750000
AGE min: 21
AGE max: 75
PAY_AMT1 min: 0
PAY_AMT2 min: 0
PAY_AMT3 min: 0
PAY_AMT4 min: 0
PAY_AMT5 min: 0
PAY_AMT6 min: 0


In [6]:
df["EDUCATION"] = df["EDUCATION"].replace([0, 5, 6], np.nan)
# Recodes 0,5,6 to NaN 

df["MARRIAGE"] = df["MARRIAGE"].replace([0], np.nan)
# Recodes 0 to NaN

df = df.dropna()
# Removes all rows with NaN values

df = df.drop(columns=["ID"])
# Remove ID column 

print("Shape after cleaning:", df.shape)
# Check whether rows were removed

print("EDUCATION after cleaning:", df['EDUCATION'].unique())
# Check whether 0,5,6 were removed

print("MARRIAGE after cleaning:", df['MARRIAGE'].unique())
# Check whether 0 was removed

Shape after cleaning: (6931, 24)
EDUCATION after cleaning: [2. 1. 3. 4.]
MARRIAGE after cleaning: [1. 2. 3.]
